In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession
from plmdb_internal import PLMDBInternal
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_request_plmdb(
    spark,
    curated_zone,
    catalog_landed,
    reference_zone,
    run_date,
    schema,
    env_prefix,
    plmdb: PLMDBInternal
    ) -> None:

    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyRequestId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        #     "IngestionTimestamp timestamp, BatchId string, PartyRequestKey string, CDI string, RequestCode int, RequestDate date,"
        #     "RequestStatusCode int, RequestStatus string, RequestTypeCode int, RequestType string, RequestCommentCode int,"
        #     "RequestCommentStatusCode int, RequestCommentStatus string, Comment string, CreatedDate timestamp, CreatedBy string,"
        #     "CreatedByChannel string, LastUpdatedDate timestamp, LockedBy string, LockedByChannel string, IsLockedFlag int,"
        #     "VendorExpsodureCode string, ExposureKey int, ExposuresSelectedFlag boolean, ExposureBalance decimal(19,4), ReportCategory string,"
        #     "ReportStatus string, ReportDate string, ReportComment string, ReportCreatedBy string, ReportCreatedByChannel string,"
        #     "ReportCreatedDate timestamp, ReportLastUpdatedDate timestamp, ReportLockedBy string, ReportLockedByChannel string, BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.PartyRequest"

        # ================== Create Delta Table ==================

        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================

        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  
                "plmdb",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== CustomerRequest ==================

        customer_request_selected_fields = [
            "CustomerRequestID",
            "CustomerRequestCommentID",
            "isLocked",
            "CustomerRequestTypeID",
            "CustomerRequestType",
            "CustomerRequestStatusID",
            "CustomerRequestStatus",
            "CustomerRequestLockedByUserChannel",
            "CustomerRequestLockedByUser",
            "CustomerRequestDateModified",
            "CustomerRequestDateCreated",
            "CustomerRequestDate",
            "CustomerRequestCreatedByUserChannel",
            "CustomerRequestCreatedByUser",
            "CustomerRequestCommentText",
            "CustomerRequestCommentStatusID",
            "CustomerRequestCommentStatus",
            "customercdi",
            "BusinessDate",
        ]

        customer_request = plmdb.customerRequest()
        customer_request = customer_request.select(customer_request_selected_fields)

        # ================== CustomerRequestLoan ==================

        customer_request_loan_selected_fields = [
            "CustomerRequestID",
            "CustomerRequestVendorLoanID",
            "CustomerRequestLoanIsSelected",
            "CustomerRequestLoanID",
            "CustomerRequestLoanBalance",
            "BusinessDate",
        ]

        customer_request_loan = plmdb.customerRequestLoan()
        customer_request_loan = customer_request_loan.select(customer_request_loan_selected_fields)

        # ================== CustomerRequestReport ==================

        customer_request_report_selected_fields = [
            "CustomerRequestID",
            "Status",
            "CustomerRequestLockedByUserChannel",
            "CustomerRequestLockedByUser",
            "CustomerRequestDateModified",
            "CustomerRequestDateCreated",
            "CustomerRequestDate",
            "CustomerRequestCreatedByUserChannel",
            "CustomerRequestCreatedByUser",
            "CustomerRequestCommentText",
            "Category",
            "BusinessDate",
        ]

        customer_request_report = plmdb.customerRequestReport()
        customer_request_report = customer_request_report.select(customer_request_report_selected_fields)

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
            "Prefix",
            "BusinessDate",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone, 
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================

        target_table_plmdb = (
            customer_request
            .join(
                customer_request_report,
                customer_request["CustomerRequestID"] == customer_request_report["CustomerRequestID"],
                "inner",
            )
            .join(
                customer_request_loan,
                customer_request_report["CustomerRequestID"] == customer_request_loan["CustomerRequestID"],
                "inner",
            )
            .join(
                portfolio,
                substring(customer_request["customercdi"], 1, 3) == portfolio["Prefix"],
                "left",
            )
        )

        # ================== Table keys ==================

        target_table_plmdb_party_requestId_primary_key_fields = [
            customer_request.CustomerRequestCommentID,
            customer_request_loan.CustomerRequestVendorLoanID,
            lit("plmdb"),
        ]

        target_table_plmdb = target_table_plmdb.withColumn(
            "PartyRequestId",
            sha2(
                concat_ws(";", *target_table_plmdb_party_requestId_primary_key_fields),
                256,
            ),
        ).withColumn(
            "PartyRequestKey",
            concat_ws(
                ";",
                customer_request.CustomerRequestCommentID,
                customer_request_loan.CustomerRequestVendorLoanID,
            ),
        )

        # ================== Final table columns & data types ==================

        result_df_plmdb = target_table_plmdb.select(
            target_table_plmdb.PartyRequestId.cast("String").alias("PartyRequestId"),
            lit("plmdb").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("String").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).cast("Timestamp").alias("IngestionTimestamp"),
            lit(None).alias("BatchId"),
            target_table_plmdb.PartyRequestKey.cast("String").alias("PartyRequestKey"),
            customer_request.customercdi.cast("String").alias("CDI"),
            customer_request.CustomerRequestID.cast("Int").alias("RequestCode"),
            customer_request.CustomerRequestDate.cast("Date").alias("RequestDate"),
            customer_request.CustomerRequestStatusID.cast("Int").alias("RequestStatusCode"),
            customer_request.CustomerRequestStatus.cast("String").alias("RequestStatus"),
            customer_request.CustomerRequestTypeID.cast("Int").alias("RequestTypeCode"),
            customer_request.CustomerRequestType.cast("String").alias("RequestType"),
            customer_request.CustomerRequestCommentID.cast("Int").alias("RequestCommentCode"),
            customer_request.CustomerRequestCommentStatusID.cast("Int").alias("RequestCommentStatusCode"),
            customer_request.CustomerRequestCommentStatus.cast("String").alias("RequestCommentStatus"),
            customer_request.CustomerRequestCommentText.cast("String").alias("Comment"),
            customer_request.CustomerRequestDateCreated.cast("Timestamp").alias("CreatedDate"),
            customer_request.CustomerRequestCreatedByUser.cast("String").alias("CreatedBy"),
            customer_request.CustomerRequestCreatedByUserChannel.cast("String").alias("CreatedByChannel"),
            customer_request.CustomerRequestDateModified.cast("Timestamp").alias("LastUpdatedDate"),
            customer_request.CustomerRequestLockedByUser.cast("String").alias("LockedBy"),
            customer_request.CustomerRequestLockedByUserChannel.cast("String").alias("LockedByChannel"),
            customer_request.isLocked.cast("Int").alias("IsLockedFlag"),
            customer_request_loan.CustomerRequestVendorLoanID.cast("String").alias("VendorExpsodureCode"),
            customer_request_loan.CustomerRequestLoanID.cast("Int").alias("ExposureKey"), # MUST BE CHANGED TO STRING-ExposureKey IS NOT OF INTEGER TYPE!
            customer_request_loan.CustomerRequestLoanIsSelected.cast("Boolean").alias("ExposuresSelectedFlag"),
            customer_request_loan.CustomerRequestLoanBalance.cast("Decimal(19,4)").alias("ExposureBalance"),
            customer_request_report.Category.cast("String").alias("ReportCategory"),
            customer_request_report.Status.cast("String").alias("ReportStatus"),
            customer_request_report.CustomerRequestDate.cast("String").alias("ReportDate"),
            customer_request_report.CustomerRequestCommentText.cast("String").alias("ReportComment"),
            customer_request_report.CustomerRequestCreatedByUser.cast("String").alias("ReportCreatedBy"),
            customer_request_report.CustomerRequestCreatedByUserChannel.cast("String").alias("ReportCreatedByChannel"),
            customer_request_report.CustomerRequestDateCreated.cast("Timestamp").alias("ReportCreatedDate"),
            customer_request_report.CustomerRequestDateModified.cast("Timestamp").alias("ReportLastUpdatedDate"),
            customer_request_report.CustomerRequestLockedByUser.cast("String").alias("ReportLockedBy"),
            customer_request_report.CustomerRequestLockedByUserChannel.cast("String").alias("ReportLockedByChannel"),
            customer_request.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Save data ==================

        (
            result_df_plmdb.write
            .format("delta")
            .clusterBy("BusinessDate")
            .mode("append")
            .saveAsTable(curated_table_path)
            )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")



In [0]:
plmdb = PLMDBInternal(spark, catalog_landed, run_date)

In [0]:
load_curated_party_request_plmdb(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix, plmdb)